# 06 - My final board-characteristic analysis

I wait for all 1,000 audited answers from each model. My pooled regression includes rating, board characteristics, frozen theme/phase encoding and model identity, with puzzle-clustered standard errors. My separate predictive trees train on 800 cases and evaluate once on 200 held-out cases. I keep the original PDFs unchanged; the revised protocol records this specification.

In [1]:
MODE = 'final'
BOOK = '06_final_analysis.ipynb'

## 1. I check audited completion and freeze inputs

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tools.sm_exceptions import PerfectSeparationWarning
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, brier_score_loss, confusion_matrix
import joblib, nbformat
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'experiments/one_move_v1/config.json').exists())
BASE=ROOT/'data/experiments/one_move_v1';CFG=ROOT/'experiments/one_move_v1'
CONFIG=json.loads((CFG/'config.json').read_text());PLAN=json.loads((CFG/'analysis_plan.json').read_text())
PREP=json.loads((BASE/'preparation.json').read_text());RECIPE=json.loads((BASE/'feature_recipe.json').read_text())
OUT=ROOT/'data/analysis/one_move_v1'/MODE;OUT.mkdir(parents=True,exist_ok=True)
FIG=ROOT/'figures/one_move_v1'/MODE

def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def save(p,value):Path(p).write_text(json.dumps(value,indent=2)+'\n',encoding='utf-8')
def code_sha(name):
    n=nbformat.read(ROOT/'notebooks'/name,as_version=4)
    return hashlib.sha256('\n\n'.join(c.source for c in n.cells if c.cell_type=='code' and 'run_control' not in c.metadata.get('tags',[])).encode()).hexdigest()
manifest=json.loads((BASE/'manifest.json').read_text())
for key,f in [('config_sha256','config.json'),('plan_sha256','analysis_plan.json'),('prompt_sha256','prompt.txt')]:assert sha(CFG/f)==manifest[key]
for key,f in [('collection_code_sha256','04_collect_answers.ipynb'),('analysis_code_sha256','06_final_analysis.ipynb'),('preparation_code_sha256','03_prepare_study.ipynb'),('early_code_sha256','05_early_results.ipynb')]:assert code_sha(f)==manifest[key]
assert sha(BASE/'puzzles.csv')==PREP['sample_sha256'] and sha(BASE/'analysis_features.csv')==PREP['feature_sha256']
assert sha(BASE/'feature_recipe.json')==PREP['recipe_sha256']
features=pd.read_csv(BASE/'analysis_features.csv',dtype={'puzzle_id':str})
assert len(features)==1000 and features.total_solver_moves.eq(1).all()
TARGET=100 if MODE=='early' else 1000
frames={};progress=[];metadata={}
for name,spec in CONFIG['models'].items():
    folder=BASE/name;path=folder/'results.csv';data=None;complete=0;target_complete=0
    if path.exists():
        audit=json.loads((folder/'audit.json').read_text())
        assert audit['passed'] and audit['sample_sha256']==PREP['sample_sha256'] and audit['results_sha256']==sha(path)
        assert audit['manifest_sha256']==sha(BASE/'manifest.json')
        result=pd.read_csv(path,dtype={'puzzle_id':str})
        assert len(result)==result.puzzle_id.nunique()==1000 and set(result.puzzle_id)==set(features.puzzle_id)
        selected=features.loc[features.pilot] if MODE=='early' else features
        cols=['puzzle_id','status','success','failure_type','answer','returned_model','trial_id']
        data=selected.merge(result[cols],on='puzzle_id',validate='one_to_one')
        complete=int(result.status.eq('completed').sum());target_complete=int(data.status.eq('completed').sum())
        if target_complete==TARGET:
            assert data.success.isin([0,1]).all()
            if MODE=='early':assert data.split.eq('train').all()
            frames[name]=data
            metadata[name]={'source_results_sha256':sha(path),'audit_sha256':sha(folder/'audit.json'),'n':len(data)}
    progress.append({'Model':spec['label'],'Completed overall':complete,'Ready cases':target_complete,'Required for this notebook':TARGET})
display(pd.DataFrame(progress))
READY=len(frames)==2
if not READY:
    save(OUT/'status.json',{'status':'waiting_for_both_pilots' if MODE=='early' else 'waiting_for_complete_collection','models':progress,'new_api_requests':0,'fits_created':False,'heldout_evaluation_performed':False})
    print('No new-study results yet. I leave the analysis pending until both models have the required audited answers. No API calls.')
if READY:
    snap_path=OUT/'snapshot.json'
    if not snap_path.exists():
        for name,d in frames.items():
            d.to_csv(OUT/(name+'_inputs.csv'),index=False)
            metadata[name]['inputs_sha256']=sha(OUT/(name+'_inputs.csv'))
        snapshot={'mode':MODE,'created_utc':datetime.now(timezone.utc).isoformat(),'sample_sha256':PREP['sample_sha256'],
                  'feature_sha256':PREP['feature_sha256'],'plan_sha256':sha(CFG/'analysis_plan.json'),'code_sha256':code_sha(BOOK),'models':metadata}
        save(snap_path,snapshot)
    else:snapshot=json.loads(snap_path.read_text())
    assert snapshot['plan_sha256']==sha(CFG/'analysis_plan.json') and snapshot['code_sha256']==code_sha(BOOK)
    assert snapshot['sample_sha256']==PREP['sample_sha256'] and snapshot['feature_sha256']==PREP['feature_sha256']
    for name in CONFIG['models']:
        p=OUT/(name+'_inputs.csv');assert sha(p)==snapshot['models'][name]['inputs_sha256']
        frames[name]=pd.read_csv(p,dtype={'puzzle_id':str})
        if MODE=='early':assert frames[name].split.eq('train').all()
    print('I use frozen inputs. Later collection cannot overwrite this snapshot.')

,Model,Completed overall,Ready cases,Required for this notebook
0,GPT-OSS 120B,100,100,1000
1,Gemini 3.5 Flash-Lite,100,100,1000


No new-study results yet. I leave the analysis pending until both models have the required audited answers. No API calls.


## 2. I fit the fixed models when supported

In [3]:
def wilson(correct,n):
    p=correct/n;z=1.96;den=1+z*z/n;center=(p+z*z/(2*n))/den
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return [float(center-half),float(center+half)]
fit=None;trees={};coefs=None
if READY:
    predictors=RECIPE['core'] if MODE=='early' else RECIPE['full']
    tree_predictors=[{'rating_500':'puzzle_rating','legal_moves_10':'legal_moves','piece_count_10':'piece_count'}.get(k,k) for k in predictors]
    pooled=pd.concat([d.assign(is_gemini=int(name=='gemini'),model_key=name) for name,d in frames.items()],ignore_index=True)
    names=predictors+['is_gemini'];X=sm.add_constant(pooled[names].astype(float),has_constant='add');y=pooled.success.astype(int)
    threshold=15 if MODE=='early' else 50
    supported=all(min(int(d.success.sum()),len(d)-int(d.success.sum()))>=threshold for d in frames.values())
    reasons=[]
    if not supported:reasons.append('Insufficient successes or failures in a model; I do not simplify the fixed model to force a fit.')
    elif np.linalg.matrix_rank(X.to_numpy())!=X.shape[1]:reasons.append('Fixed design matrix is rank deficient; no regression estimated.')
    else:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter('error',PerfectSeparationWarning)
                fit=sm.GLM(y,X,family=sm.families.Binomial()).fit(cov_type='cluster',cov_kwds={'groups':pd.factorize(pooled.puzzle_id)[0]})
            assert fit.converged and np.isfinite(fit.params).all() and np.isfinite(fit.cov_params().to_numpy()).all()
            ci=fit.conf_int()
            coefs=pd.DataFrame({'term':fit.params.index,'coefficient':fit.params.values,'odds_ratio':np.exp(fit.params.values),
                               'or_ci_low':np.exp(ci[0].values),'or_ci_high':np.exp(ci[1].values),'p_value':fit.pvalues.values})
            assert np.isfinite(coefs.select_dtypes('number').to_numpy()).all()
            coefs.to_csv(OUT/'logistic_coefficients.csv',index=False);joblib.dump(fit,OUT/'logistic_regression.joblib')
            if MODE=='final':save(OUT/'primary_rating_test.json',{'two_sided_p':float(fit.pvalues['rating_500']),'cluster':'puzzle_id','scope':'Prespecified rating association; secondary coefficients are descriptive.'})
        except Exception as error:
            fit=None;reasons.append('Regression checks failed: '+type(error).__name__)
    rows=[];metrics=[]
    for name,d in frames.items():
        correct=int(d.success.sum());rows.append({'Model':CONFIG['models'][name]['label'],'Correct':correct,'Puzzles':len(d),'Accuracy':correct/len(d),'wilson_95':wilson(correct,len(d))})
        train=d if MODE=='early' else d.loc[d.split.eq('train')]
        required=15 if MODE=='early' else 30
        if min(int(train.success.sum()),len(train)-int(train.success.sum()))<required:continue
        tree=DecisionTreeClassifier(criterion='gini',max_depth=2 if MODE=='early' else 3,min_samples_leaf=15 if MODE=='early' else 40,random_state=401)
        tree.fit(train[tree_predictors],train.success.astype(int));trees[name]=tree
        # I freeze training-only tree bytes before computing any held-out metrics.
        payload={'model':tree,'features':tree_predictors,'train_ids':sorted(train.puzzle_id),'scope':MODE}
        tree_path=OUT/(name+'_tree.joblib')
        if tree_path.exists():
            cached=joblib.load(tree_path)
            assert cached['features']==tree_predictors and cached['train_ids']==sorted(train.puzzle_id)
            assert np.array_equal(cached['model'].tree_.threshold,tree.tree_.threshold)
            assert np.array_equal(cached['model'].tree_.value,tree.tree_.value)
        else:joblib.dump(payload,tree_path)
        if MODE=='final':
            test=d.loc[d.split.eq('test')];assert len(train)==800 and len(test)==200 and not set(train.puzzle_id)&set(test.puzzle_id)
            p=tree.predict_proba(test[tree_predictors])[:,list(tree.classes_).index(1)];pred=(p>=.5).astype(int);truth=test.success.astype(int)
            majority=int(train.success.mean()>=.5);ncorrect=int((pred==truth).sum())
            metrics.append({'Model':CONFIG['models'][name]['label'],'test_n':200,'accuracy':float(accuracy_score(truth,pred)),
                'wilson_95':wilson(ncorrect,200),'balanced_accuracy':float(balanced_accuracy_score(truth,pred)),
                'roc_auc':float(roc_auc_score(truth,p)) if truth.nunique()==2 else None,'brier':float(brier_score_loss(truth,p)),
                'confusion_tn_fp_fn_tp':confusion_matrix(truth,pred,labels=[0,1]).ravel().tolist(),'majority_baseline_accuracy':float((truth==majority).mean())})
            pd.DataFrame({'puzzle_id':test.puzzle_id,'truth':truth,'probability':p,'prediction':pred}).to_csv(OUT/(name+'_heldout_predictions.csv'),index=False)
    pd.DataFrame(rows).to_csv(OUT/'accuracy.csv',index=False)
    if metrics:save(OUT/'heldout_metrics.json',metrics)
    report={'status':'analyzed','scope':MODE,'regression_fitted':fit is not None,'puzzle_clusters':TARGET,'reasons':reasons,
            'tree_models':list(trees),'heldout_evaluation_performed':MODE=='final' and bool(metrics),'new_api_requests':0,'accuracy':rows}
    save(OUT/'status.json',report);display(pd.DataFrame(rows))
    if coefs is not None:display(coefs.drop(columns='p_value') if MODE=='early' else coefs)
    for reason in reasons:print(reason)

## 3. My three figures appear after collection

In [4]:
if READY:
    FIG.mkdir(parents=True,exist_ok=True)
    heading='Pilot: 100 training puzzles per model' if MODE=='early' else 'Final: assisted one-move puzzle accuracy'
    fig,ax=plt.subplots(figsize=(8,4))
    for i,row in enumerate(rows):
        low,high=row['wilson_95'];p=row['Accuracy'];ax.bar(i,p,width=.55,color=['#2563eb','#16a34a'][i])
        ax.errorbar(i,p,yerr=[[p-low],[high-p]],fmt='none',color='black',capsize=4)
        ax.text(i,min(.96,p+.10),f"{row['Correct']}/{row['Puzzles']} ({p:.1%})",ha='center')
    ax.set(xticks=[0,1],xticklabels=[r['Model'] for r in rows],ylim=(0,1),ylabel='Official solution-move accuracy',title=heading)
    fig.tight_layout();fig.savefig(FIG/'accuracy.png',dpi=180);plt.show()
    fig,ax=plt.subplots(figsize=(8,4.5))
    if fit is not None:
        ratings=np.linspace(pooled.puzzle_rating.min(),pooled.puzzle_rating.max(),200)
        reference={k:0. for k in predictors}
        for k in RECIPE['core']:reference[k]=float(features.loc[features.split.eq('train'),k].median())
        for name,color in [('gpt_oss','#2563eb'),('gemini','#16a34a')]:
            grid=pd.DataFrame({k:np.full(200,v) for k,v in reference.items()});grid['rating_500']=(ratings-1500)/500;grid['is_gemini']=int(name=='gemini')
            grid=sm.add_constant(grid,has_constant='add')[X.columns]
            prediction=fit.get_prediction(grid).summary_frame()
            ax.plot(ratings,prediction['mean'],color=color,label=CONFIG['models'][name]['label'])
            ax.fill_between(ratings,prediction['mean_ci_lower'],prediction['mean_ci_upper'],color=color,alpha=.12)
        ax.legend();ax.set(ylim=(0,1),xlabel='Human puzzle rating',ylabel='Predicted solution-move success',title='Exploratory adjusted curves' if MODE=='early' else 'Adjusted success curves')
        ax.text(.5,-.23,'Numeric board features held at training medians; theme indicators zero and reference phase.\nOrdinary 95% bands use puzzle-clustered standard errors.',transform=ax.transAxes,ha='center',fontsize=8)
    else:ax.text(.5,.5,'Regression not supported; see status explanation.',transform=ax.transAxes,ha='center');ax.set_axis_off()
    fig.tight_layout();fig.savefig(FIG/'logistic_regression.png',dpi=180);plt.show()
    fig,axes=plt.subplots(2,1,figsize=(17,10))
    labels={'puzzle_rating':'Rating','legal_moves':'Legal moves','piece_count':'Pieces','material_balance':'Material balance'}
    for ax,name in zip(axes,CONFIG['models']):
        if name in trees:
            plot_tree(trees[name],feature_names=[labels.get(k,k.replace('theme_','').replace('phase_','Phase: ')) for k in tree_predictors],class_names=['Wrong','Correct'],filled=True,rounded=True,proportion=False,precision=2,fontsize=8,ax=ax)
        else:ax.text(.5,.5,'Insufficient class counts; no tree fitted.',transform=ax.transAxes,ha='center');ax.set_axis_off()
        ax.set_title(CONFIG['models'][name]['label'])
    fig.suptitle('Descriptive pilot trees; no held-out evaluation' if MODE=='early' else 'Final trees fitted on 800 training puzzles per model')
    fig.tight_layout();fig.savefig(FIG/'decision_trees.png',dpi=180);plt.show()
    print('I saved three figures in',FIG.relative_to(ROOT))

I make no API calls in this notebook. Missing results remain pending; I never substitute historical pilot answers. I interpret associations rather than causal effects, and accuracy applies to my balanced one-move sample under legal-choice assistance.